In [3]:
from typing import Literal, Optional, TypedDict
from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph import StateGraph, START, END
import re
from langchain_core.tools import tool

In [2]:
llm = ChatOllama(model = "gpt-oss:120b-cloud")

In [4]:
class InsuranceState(TypedDict , total = False):
    #Original customers questions
    question : str

    #Supervisor's routing decision
    selected_agent : Literal[
        "policy_agent",
        "claims_agent",
        "guidance_agent"
    ]

    #Identifiers extracted from the question
    policy_number : str
    claim_number : str

    # Information returned by a tool
    tool_result: dict

    # Final customer-friendly response
    answer : str

    error : Optional[str]


In [7]:
SUPERVISOR_PROMPT = """
You are a routing supervisor for an Insurance Claims Assistant.

Your task is to analyze the customer's question and select
exactly one agent.

Available agents:

1. policy_agent
   Select this agent for:
   - Policy coverage
   - Deductibles
   - Policy status
   - Policy expiry
   - Questions containing a policy number such as POL-1001

2. claims_agent
   Select this agent for:
   - Existing claim status
   - Claim progress
   - Claim payment status
   - Required action for an existing claim
   - Questions containing a claim number such as CLM-102

3. guidance_agent
   Select this agent for:
   - How to report an accident
   - Documents required for a claim
   - Steps to start a claim
   - General claim-process guidance

Return only one of these values:

policy_agent
claims_agent
guidance_agent

Do not include any explanation.
"""

In [ ]:
def policy_agent_node(state: InsuranceState):
    """
    Handles questions related to insurance policies,
    coverage, deductibles, policy status, and expiry.
    """

    # Get the customer's question from the shared state
    question = state.get("question", "").strip()

    # Handle missing or empty questions
    if not question:
        return {
            "error": "Customer question is missing.",
            "answer": "Please enter a policy-related question."
        }

    # Instructions for the Policy Agent
    prompt = f"""
You are a Policy Support Agent for a motor insurance company.

Your responsibilities:
- Answer general policy and coverage questions
- Explain insurance deductibles in simple language
- Explain policy status and expiry
- Provide clear and customer-friendly answers
- Never guarantee that a claim will be approved
- Never invent customer-specific policy information

Important:
The Policy Lookup Tool has not been added yet.

If the customer asks about a specific policy number,
clearly explain that you cannot retrieve that policy's
actual details yet.

Customer question:
{question}
"""

    # Send the prompt to the LLM
    response = llm.invoke(prompt)

    # Return the answer as a state update
    return {
        "answer": response.content
    }


In [9]:
def supervisor_node(state: InsuranceState):
    """
    Analyze the customer's question and select
    the appropriate specialist agent.
    """
    # question = state["question"]
    question = state.get("question", "").strip()

    if not question:
        return {
            "error": "Customer question is missing.",
            "answer": "Please enter an insurance-related question."
        }

    prompt = f"""
{SUPERVISOR_PROMPT}

Customer question:
{question}
"""

    response = llm.invoke(prompt)

    selected_agent = response.content.strip().lower()

    return {
        "selected_agent": selected_agent
    }

In [10]:
def claims_agent_node(state: InsuranceState):
    """
    Answer questions related to existing insurance claims,
    claim status, progress, and required actions.
    """

    question = state.get("question", "").strip()

    if not question:
        return {
            "error": "Customer question is missing.",
            "answer": "Please enter a question about your insurance claim."
        }

    prompt = f"""
 a Policy Support Agent for a motor insurance company.

:
 policy and coverage questions
 deductibles in es:
- Answer questions about existing claims
- Explain common claim-processing stages
- Explain possible next steps
- Avoid inventing claim information

Important:
The Claim Status Tool has not been added yet.
If the customer provides a specific claim number,
tell them that claim lookup will be added later.

Customer question:
{question}
"""

    response = llm.invoke(prompt)

    return {
        "answer": response.content
    }

In [11]:
def guidance_agent_node(state: InsuranceState):
    """
    Provide guidance about reporting an incident
    and starting an insurance claim.
    """

    question = state["question"]

    prompt = f"""
You are an Insurance Claim Guidance Agent.

Your responsibilities:
- Explain how to report an accident
- Explain how to start an insurance claim
- List the commonly required documents
- Provide clear and simple next steps
- Do not approve or reject claims

Commonly required information may include:
- Policy number
- Incident date and time
- Incident location
- Description of the incident
- Vehicle details
- Photographs of the damage
- Police report, when applicable
- Contact details of involved parties

Customer question:
{question}
"""

    response = llm.invoke(prompt)

    return {
        "answer": response.content
    }

In [12]:
def route_to_specialist(state: InsuranceState):
    """
    Read the supervisor's decision and return
    the route for the next graph node.
    """

    selected_agent = state["selected_agent"]

    if selected_agent == "policy_agent":
        return "policy"

    elif selected_agent == "claims_agent":
        return "claims"

    else:
        return "guidance"

In [14]:
graph_builder = StateGraph(InsuranceState)

In [15]:
graph_builder.add_node(
    "supervisor",
    supervisor_node
)

graph_builder.add_node(
    "policy_agent",
    policy_agent_node
)

graph_builder.add_node(
    "claims_agent",
    claims_agent_node
)

graph_builder.add_node(
    "guidance_agent",
    guidance_agent_node
)

In [18]:
graph_builder.add_edge(START,"supervisor")

In [16]:
graph_builder.add_conditional_edges(
    "supervisor",
    route_to_specialist,
    {
        "policy": "policy_agent",
        "claims": "claims_agent",
        "guidance": "guidance_agent"
    }
)

In [19]:
graph_builder.add_edge(
    "policy_agent",
    END
)

graph_builder.add_edge(
    "claims_agent",
    END
)

graph_builder.add_edge(
    "guidance_agent",
    END
)

In [20]:
insurance_graph = graph_builder.compile()

print("Insurance Claims Assistant compiled successfully.")

Insurance Claims Assistant compiled successfully.


In [21]:
policy_result = insurance_graph.invoke({
    "question": "Does policy POL-1001 cover accidental damage?"
})

print("Selected agent:", policy_result["selected_agent"])
print()
print("Answer:")
print(policy_result["answer"])

Selected agent: policy_agent

Answer:
**I’m sorry—I don’t have access to specific policy details at the moment, so I can’t say for sure whether policy POL‑1001 includes accidental‑damage coverage.**  

Here’s some general information that might help you determine if a typical motor‑insurance policy would cover accidental damage:

| What it is | How it’s usually handled |
|------------|--------------------------|
| **Accidental damage** (often called “Accidental Loss” or “Comprehensive” coverage) | Covers physical damage to your vehicle caused by an accident that is not a collision with another vehicle (e.g., hitting a tree, a pole, a ditch, or a wildlife animal). |
| **Typical inclusions** | • Collision with objects (trees, fences, etc.) <br>• Damage from overturning or rolling over <br>• Vandalism or attempted theft that results in damage |
| **Typical exclusions** | • Damage caused intentionally by the driver <br>• Mechanical breakdowns or wear‑and‑tear <br>• Damage while the vehicle

In [22]:
claim_result = insurance_graph.invoke({
    "question": "What is the status of claim CLM-102?"
})

print("Selected agent:", claim_result["selected_agent"])
print()
print("Answer:")
print(claim_result["answer"])

Selected agent: claims_agent

Answer:
I’m sorry, but I don’t have the ability to look up claim details right now. The claim‑status lookup tool is still being added, so I’m unable to provide the current status of claim **CLM‑102** at this moment. 

If you have any other questions about your policy, coverage, deductibles, or the claim‑processing steps in general, I’d be happy to help!


In [23]:
guidance_result = insurance_graph.invoke({
    "question": "What documents do I need after a car accident?"
})

print("Selected agent:", guidance_result["selected_agent"])
print()
print("Answer:")
print(guidance_result["answer"])

Selected agent: guidance_agent

Answer:
### Documents you’ll usually need after a car accident  

| Category | Typical items to gather | Why it’s needed |
|----------|------------------------|-----------------|
| **Policy information** | • Your insurance policy number (often on the ID card) <br>• Name of the insured and the insured vehicle | Helps the insurer locate your coverage quickly |
| **Incident details** | • Date and exact time of the accident <br>• Full address or GPS coordinates of the crash site | Establishes when/where the loss occurred |
| **Vehicle information** | • Year‑make‑model, VIN, license plate <br>• Current mileage (optional but helpful) | Confirms the vehicle involved and its condition |
| **Damage evidence** | • Clear photos of all exterior damage (all angles) <br>• Interior damage (dashboard, seats, airbags) <br>• Close‑up of any broken parts (glass, lights, bumper) | Visual proof is the fastest way to assess repair costs |
| **Police / official report** | • Po